# Customer Service Chatbot Training (Google Colab)

This notebook trains a DialoGPT model on the Bitext customer support dataset using three fine-tuning methods:
- **Full Fine-Tuning**: Update all model weights
- **LoRA (Low-Rank Adaptation)**: Train only low-rank adapters
- **Prefix Tuning**: Learn continuous prompts

**GPU Required**: Run this notebook on Google Colab with GPU acceleration enabled.

**Setup**: Go to Runtime → Change Runtime Type → GPU before running cells.

In [ ]:
!pip install -q transformers datasets peft evaluate torch accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00


In [ ]:
import torch
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from peft import get_peft_model, LoraConfig, PrefixTuningConfig, TaskType
import json

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

GPU Available: True
GPU Name: Tesla T4


In [ ]:
# Load dataset and check structure
print("Loading Bitext customer support dataset...")
ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split='train')
print(f"Dataset size: {len(ds)}")
print(f"Columns: {ds.column_names}")
print(f"\nExample 0:")
for k, v in ds[0].items():
    val_str = str(v)[:120] if len(str(v)) > 120 else str(v)
    print(f"  {k}: {val_str}")

# Take first 5000 for faster training (adjust as needed)
ds_train = ds.select(range(min(5000, len(ds))))
print(f"\nUsing {len(ds_train)} samples for training")

Loading Bitext customer support dataset...


Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Dataset size: 26872
Columns: ['flags', 'instruction', 'category', 'intent', 'response']

Example 0:
  flags: B
  instruction: question about cancelling order {{Order Number}}
  category: ORDER
  intent: cancel_order
  response: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the inf

Using 5000 samples for training


In [ ]:
# Dataset preprocessing
def infer_pair_fields(ds):
    """Auto-detect input and output columns."""
    cols = set(ds.column_names)
    input_candidates = ["instruction", "query", "question", "prompt", "input"]
    target_candidates = ["response", "answer", "output", "completion", "target"]

    input_field = next((c for c in input_candidates if c in cols), None)
    target_field = next((c for c in target_candidates if c in cols), None)

    if input_field and target_field:
        return input_field, target_field
    raise ValueError(f"Cannot infer fields from columns: {sorted(cols)}")

def tokenize_function(examples, tokenizer, input_field, target_field, max_length=512):
    texts = [f"{ex}\n{tgt}" for ex, tgt in zip(examples[input_field], examples[target_field])]
    encodings = tokenizer(texts, truncation=True, max_length=max_length)
    return encodings

# Setup tokenizer
model_name = "microsoft/DialoGPT-large"
print(f"Loading tokenizer: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

input_field, target_field = infer_pair_fields(ds_train)
print(f"Inferred fields: input='{input_field}', target='{target_field}'")

# Tokenize dataset
print("Tokenizing dataset...")
tokenized_ds = ds_train.map(
    lambda x: tokenize_function(x, tokenizer, input_field, target_field),
    batched=True,
    batch_size=50,
)
print(f"Tokenized {len(tokenized_ds)} examples")

Loading tokenizer: microsoft/DialoGPT-large


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Inferred fields: input='instruction', target='response'
Tokenizing dataset...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenized 5000 examples


## Training: Full Fine-Tuning

Update all model parameters. Baseline for comparison.

In [ ]:
import gc
import torch

def clear_gpu_memory():
    global model_full, model_lora, model_prefix, trainer_full, trainer_lora, trainer_prefix, total_params, trainable_params, trainable_lora, trainable_prefix
    for var in ['model_full', 'model_lora', 'model_prefix', 'trainer_full', 'trainer_lora', 'trainer_prefix', 'total_params', 'trainable_params', 'trainable_lora', 'trainable_prefix']:
        if var in globals():
            del globals()[var]
    gc.collect()
    torch.cuda.empty_cache()
    gc.collect()

clear_gpu_memory()

print("Loading model for full fine-tuning...")
model_full = AutoModelForCausalLM.from_pretrained(model_name)
model_full.config.pad_token_id = tokenizer.pad_token_id
model_full.config.tie_word_embeddings = False

total_params = sum(p.numel() for p in model_full.parameters())
trainable_params = sum(p.numel() for p in model_full.parameters() if p.requires_grad)
print(f"Full Fine-Tuning - Total params: {total_params:,}, Trainable: {trainable_params:,} (ratio: {trainable_params/total_params*100:.2f}%)")

training_args = TrainingArguments(
    output_dir="/tmp/full_finetune",
    per_device_train_batch_size=1, # Reduced to 1 to save memory
    gradient_accumulation_steps=4, # Accumulate gradients to maintain batch size
    num_train_epochs=1,
    logging_steps=50,
    save_strategy="epoch", # Save the model after each epoch
    save_total_limit=1,    # Keep only the last saved model
    fp16=False, # Changed to False to fix the 'Attempting to unscale FP16 gradients' error
)

trainer_full = Trainer(
    model=model_full,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("Starting full fine-tuning with gradient accumulation...")
trainer_full.train()
print("Full fine-tuning complete!")

Loading model for full fine-tuning...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-large
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Full Fine-Tuning - Total params: 838,359,040, Trainable: 838,359,040 (ratio: 100.00%)
Starting full fine-tuning with gradient accumulation...


Step,Training Loss
50,57703.380000
100,0.000000
150,0.000000
200,0.000000
250,0.000000
300,0.000000
350,0.000000
400,0.000000
450,0.000000
500,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Full fine-tuning complete!


## Training: LoRA (Low-Rank Adaptation)

Train only low-rank adapter weights. Much more parameter-efficient than full fine-tuning.

In [ ]:
# Fix for ImportError: Upgrade torchao to a compatible version
!pip install -q "torchao>0.16.0"

import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType

print("Loading model for LoRA...")
# Reload the base model to ensure a clean state if previous training attempts left artifacts
model_lora = AutoModelForCausalLM.from_pretrained(model_name)
model_lora.config.pad_token_id = tokenizer.pad_token_id

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
)
model_lora = get_peft_model(model_lora, lora_config)

total_params_lora = sum(p.numel() for p in model_lora.parameters())
trainable_lora = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f"LoRA - Total params: {total_params_lora:,}, Trainable: {trainable_lora:,} (ratio: {trainable_lora/total_params_lora*100:.2f}%)")

training_args_lora = TrainingArguments(
    output_dir="/tmp/lora",
    per_device_train_batch_size=1, # Reduced to 1 for memory, LoRA is efficient but large model still needs care
    gradient_accumulation_steps=4, # Accumulate gradients
    num_train_epochs=1,
    logging_steps=100,
    save_strategy="epoch", # Save the model after each epoch
    save_total_limit=1,    # Keep only the last saved model
    fp16=True,
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("Starting LoRA training...")
trainer_lora.train()
print("LoRA training complete!")

# Count trainable params for LoRA - Already calculated above for print, but keeping for clarity in comparison table
# trainable_lora = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
# print(f"LoRA - Total params: {total_params_lora:,}, Trainable: {trainable_lora:,} (ratio: {trainable_lora/total_params_lora*100:.2f}%)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.5 MB/s eta 0:00:00
Loading model for LoRA...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-large
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


LoRA - Total params: 839,833,600, Trainable: 1,474,560 (ratio: 0.18%)
Starting LoRA training...


Step,Training Loss
100,6.251002
200,4.360191
300,3.447592
400,2.991347
500,2.748291
600,2.538255
700,2.343459
800,2.268966
900,2.208063
1000,2.179217


LoRA training complete!


## Training: Prefix Tuning

Learn continuous prompts prepended to the model. Another parameter-efficient alternative.

In [ ]:
clear_gpu_memory()
print("Memory cleared. Loading model for Prefix Tuning...")

# Reload the base model to ensure a clean state
model_prefix = AutoModelForCausalLM.from_pretrained(model_name)
model_prefix.config.pad_token_id = tokenizer.pad_token_id
model_prefix.config.tie_word_embeddings = False

prefix_config = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=20,
)
model_prefix = get_peft_model(model_prefix, prefix_config)

total_params_prefix = sum(p.numel() for p in model_prefix.parameters())
trainable_prefix = sum(p.numel() for p in model_prefix.parameters() if p.requires_grad)
print(f"Prefix Tuning - Total params: {total_params_prefix:,}, Trainable: {trainable_prefix:,} (ratio: {trainable_prefix/total_params_prefix*100:.2f}%)")

training_args_prefix = TrainingArguments(
    output_dir="/tmp/prefix",
    per_device_train_batch_size=1, # Minimized batch size
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    logging_steps=50,
    save_strategy="epoch", # Save the model after each epoch
    save_total_limit=1,    # Keep only the last saved model
    fp16=True, # PEFT is safer with fp16
)

trainer_prefix = Trainer(
    model=model_prefix,
    args=training_args_prefix,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("Starting Prefix Tuning...")
trainer_prefix.train()
print("Prefix Tuning complete!")

Memory cleared. Loading model for Prefix Tuning...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-large
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prefix Tuning - Total params: 840,202,240, Trainable: 1,843,200 (ratio: 0.22%)
Starting Prefix Tuning...


Step,Training Loss
50,11.275033
100,11.024121
150,10.832216
200,10.504713
250,10.380521
300,10.164810
350,10.014280
400,9.759194
450,9.573097
500,9.375623


Prefix Tuning complete!


## Comparison: Full vs. LoRA vs. Prefix Tuning

Summary of efficiency and parameter usage across the three methods.

In [ ]:
import pandas as pd

# Create comparison table
comparison = {
    "Method": ["Full Fine-Tuning", "LoRA", "Prefix Tuning"],
    "Total Params": [f"{total_params:,}", f"{total_params:,}", f"{total_params:,}"],
    "Trainable Params": [f"{trainable_params:,}", f"{trainable_lora:,}", f"{trainable_prefix:,}"],
    "% Trainable": [
        f"{trainable_params/total_params*100:.2f}%",
        f"{trainable_lora/total_params*100:.2f}%",
        f"{trainable_prefix/total_params*100:.2f}%"
    ],
}

df_comparison = pd.DataFrame(comparison)
print("\n=== Training Methods Comparison ===")
print(df_comparison.to_string(index=False))

# Save comparison as JSON
results = {
    "model": model_name,
    "dataset": "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    "samples_used": len(tokenized_ds),
    "comparison": df_comparison.to_dict("records"),
}

print("\n✓ Training complete! All three methods successfully trained on Colab GPU.")

In [ ]:
import os
from google.colab import files

# The actual output directory used in trainer_lora was /tmp/lora
lora_path = '/tmp/lora'
zip_path = '/content/lora_weights.zip'

if os.path.exists(lora_path):
    print(f"Zipping weights from {lora_path}...")
    os.system(f"zip -r {zip_path} {lora_path}")
    files.download(zip_path)
else:
    print(f"Error: Directory {lora_path} not found. Please ensure LoRA training completed successfully.")

Zipping weights from /tmp/lora...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
from google.colab import files

full_finetune_path = '/tmp/full_finetune'
zip_path_full = '/content/full_finetune_weights.zip'

if os.path.exists(full_finetune_path):
    print(f"Zipping weights from {full_finetune_path}...")
    os.system(f"zip -r {zip_path_full} {full_finetune_path}")
    files.download(zip_path_full)
else:
    print(f"Error: Directory {full_finetune_path} not found. Please ensure full fine-tuning completed successfully.")

Zipping weights from /tmp/full_finetune...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
from google.colab import files

prefix_path = '/tmp/prefix'
zip_path_prefix = '/content/prefix_tuning_weights.zip'

if os.path.exists(prefix_path):
    print(f"Zipping weights from {prefix_path}...")
    os.system(f"zip -r {zip_path_prefix} {prefix_path}")
    files.download(zip_path_prefix)
else:
    print(f"Error: Directory {prefix_path} not found. Please ensure Prefix Tuning completed successfully.")

Zipping weights from /tmp/prefix...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>